# Tugas 2


## Memecah dataset iris menjadi 4 bagian


### Data (id, sepal length : di posgresql local)


In [ ]:
# a. postgres local
import psycopg2
import pandas as pd
# Connection parameters, yours will be different
param_dic = {
    "host"      : "localhost",
    "database"  : "sepallength",
    "user"      : "postgres",
    "password"  : "natul",
    "port"      : 5432
}
def connect(params_dic):
    """ Connect to the PostgreSQL database server """
    conn = None
    try:
        # connect to the PostgreSQL server
        print('Connecting to the PostgreSQL database...')
        conn = psycopg2.connect(**params_dic)
    except (Exception, psycopg2.DatabaseError) as error:
        print(error)
        sys.exit(1) 
    print("Connection successful")
    return conn

def postgresql_to_dataframe(conn, select_query, column_names):
    """
    Tranform a SELECT query into a pandas dataframe
    """
    cursor = conn.cursor()
    try:
        cursor.execute(select_query)
    except (Exception, psycopg2.DatabaseError) as error:
        print("Error: %s" % error)
        cursor.close()
        return 1
    
    # Naturally we get a list of tupples
    tupples = cursor.fetchall()
    cursor.close()
    
    # We just need to turn it into a pandas dataframe
    df = pd.DataFrame(tupples, columns=column_names)
    return df

# Connect to the database
conn = connect(param_dic)

column_names = ["id", "sepallength"]

df_sl = postgresql_to_dataframe(conn, "select * from iris", column_names)


### Data (id, sepal width  	: di postgeql elephantsql)


In [ ]:
# b. postgres online
import psycopg2
import pandas as pd
# Connection parameters, yours will be different
param_dic = {
    "host"      : "floppy.db.elephantsql.com",
    "database"  : "feeoezff",
    "user"      : "feeoezff",
    "password"  : "qiXzz4k6n-VgN6A9R7hvBZz8C965K80y"
}
def connect(params_dic):
    """ Connect to the PostgreSQL database server """
    conn = None
    try:
        # connect to the PostgreSQL server
        print('Connecting to the PostgreSQL database...')
        conn = psycopg2.connect(**params_dic)
    except (Exception, psycopg2.DatabaseError) as error:
        print(error)
        sys.exit(1) 
    print("Connection successful")
    return conn

def postgresql_to_dataframe(conn, select_query, column_names):
    """
    Tranform a SELECT query into a pandas dataframe
    """
    cursor = conn.cursor()
    try:
        cursor.execute(select_query)
    except (Exception, psycopg2.DatabaseError) as error:
        print("Error: %s" % error)
        cursor.close()
        return 1
    
    # Naturally we get a list of tupples
    tupples = cursor.fetchall()
    cursor.close()
    
    # We just need to turn it into a pandas dataframe
    df = pd.DataFrame(tupples, columns=column_names)
    return df

# Connect to the database
conn = connect(param_dic)

column_names = ["id", "sepalwidth"]

df_sw = postgresql_to_dataframe(conn, "select * from irissepalwith", column_names)


### Data (id, petal length 	: di mysql local)

In [ ]:
# c. mysql local

import mysql.connector as sql

import pandas as pd

db_connection = sql.connect(host='localhost', database='petallength', user='root', password='')

db_cursor = db_connection.cursor()

db_cursor.execute('SELECT * FROM iris')

table_rows = db_cursor.fetchall()
colom_name = ["id", "petallength"]
df_pl = pd.DataFrame(table_rows,columns=colom_name)

### Data (id, petal width , class  : di sql server local)

In [ ]:
# d. sql server local
import pyodbc 
import pandas as pd
cnxn_str = ("Driver={odbc driver 17 for sql server};"
            "Server=LAPTOP-MU7C7EQP\SQLEXPRESS;"
            "Database=petalwidth;"
            "Trusted_Connection=yes;")
cnxn = pyodbc.connect(cnxn_str)
cursor = cnxn.cursor()	
cursor.execute("SELECT * FROM petalwidth") 
row = cursor.fetchall() 
cursor.close()
data=[]
for i in row:
    data.append([i[0].strip('"'),i[1].strip('"'),i[2]])
df_sqlServer = pd.DataFrame(data, columns=['id','petalwidth','class'])

df_pw = df_sqlServer

## Menggabungkan 4 database


In [ ]:
data = pd.concat([df_sl,df_sw,df_pl,df_pw], axis=1)

## Menghapus colom id


In [ ]:
data.drop(columns="id",inplace=True)

## Pre-prosessing


In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB

In [ ]:
#preprosessing

X=data.iloc[:,0:4].values
y=data.iloc[:,4].values

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=0)

## Metode KNN


In [ ]:
#knn

knn_clf = KNeighborsClassifier(n_neighbors=9)
knn_clf.fit(X_train,y_train)

y_pred = knn_clf.predict(X_test)

accuracy_knn=round(accuracy_score(y_test,y_pred),3)
print(accuracy_knn)

## Metode Naive Bayes


In [ ]:
# naive bayes

gaussian = GaussianNB()
gaussian.fit(X_train, y_train)
YY_pred = gaussian.predict(X_test)
accuracy_nb=round(accuracy_score(y_test,YY_pred)* 100, 2)
print(accuracy_nb)

## Model KNN


In [ ]:
# buat model knn

import pickle

with open('knnB_pickle','wb') as r:
    pickle.dump(knn_clf,r)


## Model Naive Bayes


In [ ]:
#buat model naive bayes
import pickle

with open('nb1_pickle','wb') as r:
    pickle.dump(gaussian,r)